In [1]:
from pathlib import Path
import re

import pandas as pd

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
results_dir = Path("results")
input_csv = results_dir / "ctr_summary.csv"

output_excel = results_dir / "ctr_results_summary.xlsx"
output_cfd_csv = results_dir / "ctr_vs_voltage_cfd.csv"

if not input_csv.is_file():
    raise FileNotFoundError(
        f"Summary CSV not found: {input_csv}\n"
        "Run scripts/summarize_results.py first."
    )

# ------------------------------------------------------------------
# Load complete summary
# ------------------------------------------------------------------
summary = pd.read_csv(input_csv)

required_columns = {
    "run",
    "method",
    "ctr_ps",
    "ctr_error_ps",
}

missing = required_columns.difference(summary.columns)
if missing:
    raise ValueError(
        "The summary CSV is missing required columns: "
        + ", ".join(sorted(missing))
    )


def extract_voltage(run_name: str) -> float:
    """
    Extract the bias voltage from names such as:
        44V-370mV
        45V-400mV
        results/46V-440mV
    """
    match = re.search(r"(\d+(?:\.\d+)?)\s*V", str(run_name), flags=re.IGNORECASE)

    if match is None:
        return float("nan")

    return float(match.group(1))


summary["voltage_V"] = summary["run"].map(extract_voltage)

# Convert numeric columns safely
numeric_columns = [
    "voltage_V",
    "parameter_value",
    "ctr_ps",
    "ctr_error_ps",
    "mean_ps",
    "mean_error_ps",
    "sigma_ps",
    "sigma_error_ps",
    "chi2",
    "ndof",
    "chi2_ndof",
    "fit_low_ps",
    "fit_high_ps",
    "n_total",
    "n_selected",
    "n_rejected",
    "n_valid",
    "n_fit",
    "crossing_efficiency",
]

for column in numeric_columns:
    if column in summary.columns:
        summary[column] = pd.to_numeric(summary[column], errors="coerce")

summary = summary.sort_values(
    ["voltage_V", "method"],
    na_position="last",
).reset_index(drop=True)

# ------------------------------------------------------------------
# CFD-only table
# One row per voltage.
# If duplicate CFD rows exist, retain the one with the lowest CTR.
# ------------------------------------------------------------------
cfd = summary[
    summary["method"].astype(str).str.upper().eq("CFD")
].copy()

cfd = cfd[
    cfd["voltage_V"].notna()
    & cfd["ctr_ps"].notna()
].copy()

if "success" in cfd.columns:
    success = (
        cfd["success"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )
    cfd = cfd[success].copy()

cfd = (
    cfd.sort_values(["voltage_V", "ctr_ps"])
    .drop_duplicates(subset="voltage_V", keep="first")
    .sort_values("voltage_V")
    .reset_index(drop=True)
)

cfd_compact = cfd[
    [
        "voltage_V",
        "ctr_ps",
        "ctr_error_ps",
    ]
].rename(
    columns={
        "voltage_V": "voltage_V",
        "ctr_ps": "best_ctr_ps",
        "ctr_error_ps": "best_ctr_error_ps",
    }
)

# Save compact CSV
cfd_compact.to_csv(
    output_cfd_csv,
    index=False,
    float_format="%.6g",
)

# ------------------------------------------------------------------
# Save Excel workbook
# ------------------------------------------------------------------
with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    summary.to_excel(
        writer,
        sheet_name="All results",
        index=False,
    )

    cfd.to_excel(
        writer,
        sheet_name="Best CFD details",
        index=False,
    )

    cfd_compact.to_excel(
        writer,
        sheet_name="CTR vs voltage",
        index=False,
    )

    # Basic formatting
    workbook = writer.book

    for sheet_name in workbook.sheetnames:
        worksheet = workbook[sheet_name]

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        # Header formatting
        for cell in worksheet[1]:
            cell.font = cell.font.copy(bold=True)
            cell.alignment = cell.alignment.copy(horizontal="center")

        # Controlled column widths
        for column_cells in worksheet.columns:
            column_letter = column_cells[0].column_letter

            maximum_length = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in column_cells
            )

            worksheet.column_dimensions[column_letter].width = min(
                max(maximum_length + 2, 10),
                28,
            )

    # Numeric formatting for compact CFD sheet
    worksheet = workbook["CTR vs voltage"]

    for cell in worksheet["A"][1:]:
        cell.number_format = "0.0"

    for column in ("B", "C"):
        for cell in worksheet[column][1:]:
            cell.number_format = "0.0"

print(f"Complete Excel summary saved to:\n  {output_excel.resolve()}")
print(f"\nCFD voltage/CTR CSV saved to:\n  {output_cfd_csv.resolve()}")

display(cfd_compact)

Complete Excel summary saved to:
  C:\Users\aless\Desktop\UChicago\Prj_3\python_waveform_analysis\results\ctr_results_summary.xlsx

CFD voltage/CTR CSV saved to:
  C:\Users\aless\Desktop\UChicago\Prj_3\python_waveform_analysis\results\ctr_vs_voltage_cfd.csv


C:\Users\aless\AppData\Local\Temp\ipykernel_19960\3082035851.py:175: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.font = cell.font.copy(bold=True)
C:\Users\aless\AppData\Local\Temp\ipykernel_19960\3082035851.py:176: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.alignment = cell.alignment.copy(horizontal="center")


,voltage_V,best_ctr_ps,best_ctr_error_ps
0,45.0,64.220770,0.804187
1,46.0,63.121808,0.742333
2,47.0,62.955131,0.719790
3,48.0,62.813485,0.798787
4,49.0,60.521395,0.775291
